# Deep Learning
## Summative assessment
### Coursework 2

#### Instructions

This coursework is released on **Tuesday 11th March 9.00** and is due by **Tuesday 1st April 9.00**. It is worth **50%** of your overall mark. There are 4 questions in this assessment, and a total of 50 marks are available. **You should attempt to answer all questions.** In addition to the total number of marks shown per question below, an additional 5 marks is available for presentation and clarity/quality of code.

This assessment assesses your ability to implement and analyse modifications to the variational autoencoder (VAE) algorithm. In particular, you will explore techniques for improving the posterior approximation and improving estimates of the model evidence.

You can make imports as and when you need them throughout the notebook, and add code cells where necessary. Make sure your notebook executes correctly in sequence before submitting.

#### Submission instructions

The submission for this assessment will consist of a notebook (.ipynb file) and PDF report.

Ensure your notebook executes correctly in order. Save your notebook .ipynb file **after you have executed it** (so that outputs are all showing). It is recommended to also submit a PDF/HTML copy of your executed notebook, in case the .ipynb file is corrupted for some reason.

Upload a zip file containing your notebook and separate PDF/HTML file(s) by the deadline above.

This assignment must be attempted individually; your submission must be your own, unaided work. Candidates are prohibited from discussing assessed coursework, and must abide by [Imperial College’s rules](https://www.imperial.ac.uk/media/imperial-college/administration-and-support-services/registry/academic-governance/public/academic-policy/academic-integrity/Examination-and-assessments---academic-integrity.pdf) regarding academic integrity and plagiarism. Unless specifically authorised within the assignment instructions, the submission of output from [generative AI tools](https://www.imperial.ac.uk/about/leadership-and-strategy/provost/vice-provost-education/generative-ai-tools-guidance/) (e.g., ChatGPT) for assessed coursework is prohibited. Violations will be treated as an examination offence. Enabling other candidates to plagiarise your work constitutes an examination offence. To ensure quality assurance is maintained, departments may choose to invite a random selection of students to an ‘authenticity interview’ on their submitted assessments.

In [14]:
# You will need the following imports for this assessment. You can make additional imports when you need them

import keras
from keras import ops
import numpy as np
import matplotlib.pyplot as plt

from keras.api.models import Sequential, Model
from keras.api.layers import Input, Dense

#### Binarized MNIST Dataset

This assessment makes use of a specific binarization of the MNIST images. This dataset is frequently used to evaluate generative models of images, so labels are not provided.

The dataset was first introduced in the following paper:

* Salakhutdinov, R. and Murray, I. (2008), "On the quantitative analysis of deep belief networks", in *Proceedings of the 25th international conference on Machine learning*, 892-879.

This dataset is available from the [TensorFlow Datasets](https://www.tensorflow.org/datasets) library in prepared train/validation/test splits, and can be loaded into TF Dataset objects by running the following cell.

In [15]:
# Load a binarised version of MNIST

import tensorflow_datasets as tfds

train_ds, val_ds, test_ds = tfds.load('binarized_mnist', data_dir='data', split=['train', 'validation', 'test'])
print(train_ds)
print(train_ds.element_spec)

<_PrefetchDataset element_spec={'image': TensorSpec(shape=(28, 28, 1), dtype=tf.uint8, name=None)}>
{'image': TensorSpec(shape=(28, 28, 1), dtype=tf.uint8, name=None)}


#### Variational autoencoder

Your task in this assessment is to experiment with and analyse variants of the variational autoencoder algorithm. This will involve modifications to the objective function and the variational posterior. 

Throughout this assessment you should use Keras model subclassing and the high level `compile` and `fit` APIs to train your model using the VAE algorithm. The following base `VAE` class is provided for you (taken from the week 9 lecture notes). In the following questions you should subclass from this base class where possible, in order to reduce repeated code. You are free to override the provided methods as needed.

In [16]:
from keras.api.metrics import Mean
import tensorflow as tf
import torch


class VAE(Model):

    def __init__(self, encoder, decoder, num_mc_samples=1, **kwargs):
        """
        You should override this method as necessary in your implementations.
        """
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.loss_metric = Mean(name='loss')
        self.nll_metric = Mean(name='nll')
        self.kl_metric = Mean(name='kl')
        self.pi = ops.array(np.pi)
        self.L = num_mc_samples
        
    def compute_losses(self, data):
        """
        This method should compute and return the loss, kl_loss and nll_loss.
        You should override this method as necessary in your implementations.
        """
        z_mean, z_log_var = self.encoder(data)
        kl_loss = 0.5 * ops.sum((ops.square(z_mean) + ops.exp(z_log_var) - 1 - z_log_var), axis=-1)
        kl_loss = ops.mean(kl_loss)
        
        epsilon = keras.random.normal(ops.shape(z_mean))
        z_std = ops.exp(0.5 * z_log_var)
        z_sample = z_mean + (z_std * epsilon)
        
        x_mean, x_log_std = self.decoder(z_sample)
        log_Z = 0.5 * ops.log(2 * self.pi)
        nll_loss = 0.5 * ops.square((data - x_mean) / ops.exp(x_log_std)) + x_log_std + log_Z
        nll_loss = ops.mean(ops.sum(nll_loss, axis=[-1, -2]))

        loss = kl_loss + nll_loss
        return loss, kl_loss, nll_loss

    def call(self, inputs):
        """
        This method should compute the approximate posterior using the encoder, 
        and draw a single sample to pass through the decoder.
        You should override this method as necessary in your implementations.
        """
        z_mean, z_log_var = self.encoder(inputs)
        epsilon = keras.random.normal(ops.shape(z_mean))
        z_std = ops.exp(0.5 * z_log_var)
        z_sample = z_mean + (z_std * epsilon)
        return self.decoder(z_sample)

    def train_step(self, data):
        if keras.config.backend() == 'tensorflow':
            with tf.GradientTape() as tape:
                loss, kl_loss, nll_loss = self.compute_losses(data)
                loss = ops.mean(loss)
            grads = tape.gradient(loss, self.trainable_weights)
            self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        else:
            assert keras.config.backend() == 'torch'
            self.zero_grad()
            loss, kl_loss, nll_loss = self.compute_losses(data)
            loss = ops.mean(loss)

            loss.backward()

            gradients = [v.value.grad for v in self.trainable_weights]    
            with torch.no_grad():
                self.optimizer.apply(gradients, self.trainable_weights)
            
        self.loss_metric.update_state(loss)
        self.nll_metric.update_state(nll_loss)
        self.kl_metric.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        loss, kl_loss, nll_loss = self.compute_losses(data)
        loss = ops.mean(loss)
        self.loss_metric.update_state(loss)
        self.nll_metric.update_state(nll_loss)
        self.kl_metric.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

    @property
    def metrics(self):
        return [self.loss_metric, self.nll_metric, self.kl_metric]

### Question 1 (Total 12 marks)

a) Using the `train_ds`, `val_ds` and `test_ds` objects returned above, prepare your Datasets ready for training and evaluating your model that will be trained using the VAE algorithm. Each Dataset should return a single Tensor consisting of a batch of binarised MNIST images.

**(2 marks)**

In [17]:
def prepare_dataset(ds, batch_size: int = 64, shuffle: bool = False):
    '''
    Prepare the dataset `ds` for training or evaluation.
    Args:
        ds: A tensorflow PrefetchDataset
        batch_size: The batch size to partition the dataset into
        shuffle: If True, the dataset is shuffled

    Returns: Tensorflow DataSet
    '''
    ds = ds.map(lambda x: tf.cast(x['image'], 'float32'))  # originally tf.uint8
    if shuffle:
        ds = ds.shuffle(buffer_size=len(ds))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def get_datasets(batch_size: int = 64):
    '''
    Load and preprocess the MNIST dataset(s)
    Args:
        batch_size: The batch size

    Returns: Tensorflow dataset or tuple of Tensorflow datasets
    '''
    train, val, test = tfds.load('binarized_mnist', data_dir='data', split=['train', 'validation', 'test'])
    train = prepare_dataset(train, batch_size=batch_size, shuffle=True)
    val = prepare_dataset(val, batch_size=batch_size, shuffle=False)
    test = prepare_dataset(test, batch_size=batch_size, shuffle=False)
    return train, val, test

In [1]:
train, val, test = get_datasets()
print(len(train))
print(len(val))
print(len(test))
print(type(train))
data = next(iter(train.take(1)))
print(data.shape)
print(data.dtype)


NameError: name 'get_datasets' is not defined

b) Implement the VAE algorithm to learn a generative model of the binarized MNIST dataset. In particular, you should define encoder and decoder networks, and write a `VAEDiagonal` class that implements the training update and evaluation steps using the ELBO $\mathcal{L}_{VAE}(\theta, \phi; {x})$ as the objective function that we wish to maximise:

$$
\mathcal{L}_{VAE}(\theta, \phi; {x}) := 
 \mathbb{E}_{z\sim q_\phi({z}\mid {x})} \left[ \log p_\theta({x}, {z}) - \log q_\phi({z}\mid {x}) \right] \le \log p_\theta(x).
$$

Your implementation should follow the specifications:

* The `VAEDiagonal` class should subclass from the `VAE` class above
* The encoder network should use a fully factorised (diagonal) Gaussian for the approximate posterior $q_\phi(z \mid x)$
* The latent space should have dimension 2
* Your implementation should be entirely using Keras
* The ELBO objective should be approximated using the second form of the SGVB estimator:
$$
\hat{\mathcal{L}}^B(\theta,\phi;x)[q_\phi] := \frac{1}{L} \sum_{j=1}^L \log p_\theta(x\mid z^{(j)}) − D_{KL}(q_\phi(z\mid x) || p_\theta(z)),
$$
  * You should set $L=1$ in the model that you train (use a single Monte Carlo sample)
* The likelihood distribution $p_\theta(x\mid z^{(j)})$ should be chosen appropriately for the binary data
* Train your model using the high-level Keras API, and evaluate your trained model on the test set, clearly displaying the loss obtained on the test set.

You should aim to obtain final trained encoder and decoder networks that achieve an ELBO of at least -150 on the test set

_Make sure to save your model as it will be used again in later questions._

**(4 marks)**

### Solution

For a fully factorised Gaussian posterior $q_\phi(z \mid x)$, we have that

$$
q_\phi(z \mid x) = \prod_{i=1}^2 \mathcal{N}(z_i \mid \mu_i, \sigma_i^2)
$$

where $\mu_i$ and $\sigma_i^2$ are the mean and variance of the $i$th dimension of the latent space. The KL divergence term in the ELBO is given by


$$
\begin{align}
D_{KL}(q_\phi(z\mid x) || p_\theta(z)) = \frac{1}{2}\left[ \mu_q^T\mu_q + \sum_{i=1}^l (\sigma_q)_i - l - \log \prod_{i=1}^l (\sigma_q)_i \right], \tag{1}
\end{align}
$$

where $p_\theta(z) = \mathcal{N}(0, I)$ is the standard Gaussian prior, so $\mu_{0,i} = 0$ and $\sigma_{0,i}^2 = 1$.

The likelihood term in the ELBO corresponds to the reconstruction term $\log p_\theta(x \mid z)$ and must be chosen according to the type of observed data. Since we are using the binarized MNIST dataset, the pixel values are either 0 or 1, which suggests a Bernoulli likelihood is appropriate.
In this case, we assume that the likelihood $p_\theta(x \mid z)$ is modeled as a product of independent Bernoulli distributions:

$$
p_\theta(x \mid z) = \prod_{i=1}^D \text{Bernoulli}(x_i \mid \hat{x}_i)
$$

where $\hat{x}_i$ is the output of the decoder network for pixel $i$ after applying a sigmoid activation to ensure $\hat{x}_i \in (0,1)$.

This leads to the binary cross-entropy loss:

$$
\log p_\theta(x | z) = \text{BCE}(x, \hat{x}) = -\frac{1}{D} \sum_{i=1}^D \left( x_i \log(\hat{x}_i) + (1 - x_i) \log(1 - \hat{x}_i) \right) \tag{2}
$$

Here, $\hat{x}$ is obtained by decoding the sampled latent variable $z \sim q_\phi(z \mid x)$ through a decoder neural network with a final sigmoid activation.

Then, the ELBO becomes:

$$
\mathcal{L}(\theta, \phi; x) \approx -\text{BCE}(x, \hat{x}) - D_{KL}(q_\phi(z \mid x) \,\|\, p_\theta(z)) \tag{3}
$$
where $\text{BCE}(x, \hat{x})$ is given by (2). The KL divergence term is given by (1).

Note that (3) is the ELBO computed for a single sample image. In practice, one averages over multiple samples in the batch (in addition to the Monte Carlo samples).



### Comments on the implementation

Using tensorflow as the backend resulted in instabilities that made it impossible to train the model. This issue was not present when training with `torch` as the backend or if we set the argument `run_eagerly = False` when compiling the model. It appears that the issue is to do with the M1 Max architecture and how it handles the `tensorflow` DAG. `run_eagerly = True` forces the graph to be evaluated at each step which makes the training very slow but it resolves the instabilities.

Prior to understanding the root cause of the issue, we made numerous attempts to resolve the numerical instabilities. Part of this effort was to implement the BCE loss function from scratch, instead of using the `keras` implementation, and manually clip anything that was passed into a log or exponential.
We also introduces an instance variable that specifies whether the decoder outputs probabilities or logits.

In [ ]:
class VAEDiagonal(VAE):

    def __init__(self, encoder, decoder, num_mc_samples=1, from_logits: bool = False, **kwargs):
        """
        You should override this method as necessary in your implementations.
        """
        super().__init__(encoder=encoder, decoder=decoder, num_mc_samples=num_mc_samples, **kwargs)
        self.from_logits = from_logits

    def _sample_z(self, z_mean, z_log_var):
        '''
        Sample z using the mean and standard deviation output from the decoder and Gaussian
        noise epsilon ~ N(0, 1) (l dimensional)

            z = mu_q + std_q * epsilon
        Args:
            z_mean: The mean of z, obtained using the encoder
            z_log_var: The log variance of z, obtained using the decoder

        Returns: An (L, l) tensor, where l is the dimension of the latent space and
        L is the number of Monte Carlo samples required
        '''
        batch_size, latent_dim = ops.shape(z_mean)

        # Sample epsilon ~ N(0, 1)
        epsilon = keras.random.normal(shape=(self.L, batch_size, latent_dim))

        # Reparameterisation trick
        z_std = ops.exp(0.5 * z_log_var)
        z_samples = z_mean + z_std * epsilon  # shape: (L, B, l)

        return z_samples

    def compute_losses(self, data):
        '''
        Compute the losses. The KL Loss term is the same as the VAE implementation but
        the reconstruction loss will now be the log likelihood of the Bernoulli distribution

        As with the base class, the encoder outputs the mean and log variance of z.
        The decoder outputs the probability $p$ of the Bernoulli distribution
        Args:
            data: a minibatch of the data

        Returns: the total loss, KL loss and negative log-likelihood loss (scalars)
        '''
        eps = 1e-7  # const used for numerical stability (logs)

        # Encode the data -> image shape to latent dim
        z_mean, z_log_var = self.encoder(data)  # (batch, latent_dim)
        batch_size, latent_dim = ops.shape(z_mean)

        # KL divergence term (analytical)
        kl_per_sample = 0.5 * ops.sum(
            ops.square(z_mean) + ops.exp(z_log_var) - 1 - z_log_var,
            axis=-1  # sum over latent dim
        )
        kl_loss = ops.mean(kl_per_sample)  # scalar

        # Sample z \sim q(z|x)
        z_samples = self._sample_z(z_mean, z_log_var)  # (MC samples, batch, latent_dim)
        z_samples = ops.reshape(z_samples, (self.L * batch_size, latent_dim))  # (MC Samples * batch, latent_dim)

        # Decode: returns logits or probabilities
        x_pred = self.decoder(z_samples)  # (MC samples * batch, H, W, C)
        _, H, W, C = ops.shape(data)

        # Expand ground truth to match shape
        x_true = ops.expand_dims(data, axis=0)  # (1, batch, H, W, C)
        x_true = ops.repeat(x_true, self.L, axis=0)  # (MC samples, batch, H, W, C)
        x_true = ops.reshape(x_true, ops.shape(x_pred))

        if self.from_logits:
            # --- BCE Loss ---
            # Use Keras' numerically stable BCE (handles logits internally)
            bce_per_pixel = keras.losses.binary_crossentropy(
                y_true=x_true,
                y_pred=x_pred,
                from_logits=True
            )  # shape: (L * B, H, W)
            # Sum over pixel dims (H, W) -> total BCE per image
            bce_per_image = ops.sum(bce_per_pixel, axis=[-1, -2])  # shape: (L * B,)

        else:
            # Clip probs to avoid log(0)
            x_pred = ops.clip(x_pred, eps, 1. - eps)

            # COMPUTE BCE Explicitly (using binary cross entropy loss from keras was leading to instabilities for some reason
            bce_per_pixel = -(
                x_true * ops.log(x_pred) +
                (1. - x_true) * ops.log(1. - x_pred)
            )  # shape: (MC samples * batch, H, W, C)

            # Sum over pixel dimensions (H, W, C) -> total BCE per image
            bce_per_image = ops.sum(bce_per_pixel, axis=[-1, -2, -3])  # shape: (MC samples, batch)

        # Average over MC and batch
        nll_loss = ops.mean(bce_per_image)
        total_loss = kl_loss + nll_loss

        return total_loss, kl_loss, nll_loss


In [21]:
# Define the decoder
import keras.src.layers
from keras import ops
from keras.api.models import Model
from keras.api.layers import (Input, Dense, Reshape, Flatten, Conv2D, MaxPool2D, Conv2DTranspose, BatchNormalization,
                              ReLU, UpSampling2D)
from typing import Tuple

REGULARIZER = keras.regularizers.L2()
INITIALIZER_RELU = keras.initializers.HeUniform()
INITIALIZER_SIGMOID = keras.initializers.GlorotNormal()


def get_decoder_v0(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the decoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    (img_h, img_w, img_c) = image_shape
    inputs = Input(shape=(latent_dim,))
    h = Dense(256, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(inputs)
    h = Dense(img_h * img_w * img_c, kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_SIGMOID,
              activation='sigmoid')(h)
    outputs = Reshape((img_h, img_w, img_c))(h)
    return Model(inputs=inputs, outputs=outputs, name='decoder')


def get_encoder_v0(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the encoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    (img_h, img_w, img_c) = image_shape
    inputs = Input(shape=(img_h, img_w, img_c))
    h = Flatten()(inputs)
    h = Dense(256, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(2 * latent_dim, kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    z_mean, z_log_var = ops.split(h, indices_or_sections=2, axis=-1)

    return Model(inputs=inputs, outputs=[z_mean, z_log_var], name='encoder')


def get_decoder_v1(image_shape: Tuple[int, int], latent_dim: int):
    '''
    Get the decoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    (img_h, img_w, img_c) = image_shape
    inputs = Input(shape=(latent_dim,))
    h = Dense(64, activation='leaky_relu', kernel_regularizer=REGULARIZER,
              kernel_initializer=INITIALIZER_RELU)(inputs)
    h = Dense(128, activation='leaky_relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(256, activation='leaky_relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(img_h * img_w * img_c, activation='sigmoid', kernel_initializer=INITIALIZER_SIGMOID)(h)
    output = Reshape((img_h, img_w, img_c))(h)
    return Model(inputs=inputs, outputs=output, name='decoder')


def get_encoder_v1(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the encoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    (img_h, img_w, img_c) = image_shape
    inputs = Input(shape=(img_h, img_w, img_c))
    h = Flatten()(inputs)
    h = Dense(256, activation='leaky_relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(128, activation='leaky_relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(64, activation='leaky_relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(2 * latent_dim, kernel_regularizer=REGULARIZER)(h)
    z_mean, z_log_var = ops.split(h, indices_or_sections=2, axis=-1)
    return Model(inputs=inputs, outputs=[z_mean, z_log_var], name='encoder')


def get_decoder_v2(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the decoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    # output size for Conv2D transpose:
    # out = (in − 1) x stride − 2 x padding + kernel_size + output_padding
    # (img_h, img_w, img_c) = image_shape
    inputs = Input(shape=(latent_dim,))
    # h = Dense(64, activation='relu')(inputs)
    h = Dense(7 * 7 * 64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(
        inputs)
    h = Reshape((7, 7, 64))(h)
    h = Conv2DTranspose(32, 3, strides=2, activation='relu', padding='same')(h)  # 7 → 14
    outputs = Conv2DTranspose(1, 3, strides=2, activation='sigmoid', padding='same')(h)
    return Model(inputs=inputs, outputs=outputs, name='decoder')


def get_encoder_v2(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the encoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''

    inputs = Input(shape=image_shape)
    h = Conv2D(32, 3, strides=2, activation='relu', padding='same')(inputs)  # 28 → 14
    h = Conv2D(64, 3, strides=2, activation='relu', padding='same')(h)  # 14 → 7
    h = Flatten()(h)
    h = Dense(2 * latent_dim)(h)
    z_mean, z_log_var = ops.split(h, indices_or_sections=2, axis=-1)
    return Model(inputs=inputs, outputs=[z_mean, z_log_var], name='encoder')


def get_decoder_v3(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the decoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    # output size for Conv2D transpose:
    # out = (in − 1) x stride − 2 x padding + kernel_size + output_padding
    # (img_h, img_w, img_c) = image_shape
    inputs = Input(shape=(latent_dim,))
    # h = Dense(64, activation='relu')(inputs)
    h = Dense(7 * 7 * 64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(
        inputs)
    h = Reshape((7, 7, 64))(h)
    h = Conv2DTranspose(32, 2, strides=2, activation='relu', padding='same')(h)  # 7 → 14
    outputs = Conv2DTranspose(1, 2, strides=2, activation='sigmoid', padding='same')(h)
    return Model(inputs=inputs, outputs=outputs, name='decoder')


def get_encoder_v3(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the encoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''

    inputs = Input(shape=image_shape)
    h = Conv2D(32, 2, strides=2, activation='relu', padding='same')(inputs)  # 28 → 14
    h = Conv2D(64, 2, strides=2, activation='relu', padding='same')(h)  # 14 → 7
    h = Flatten()(h)
    h = Dense(2 * latent_dim)(h)
    z_mean, z_log_var = ops.split(h, indices_or_sections=2, axis=-1)
    return Model(inputs=inputs, outputs=[z_mean, z_log_var], name='encoder')


def get_decoder_v4(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the decoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    # output size for Conv2D transpose:
    # out = (in − 1) x stride − 2 x padding + kernel_size + output_padding
    (img_h, img_w, img_c) = image_shape
    inputs = Input(shape=(latent_dim,))
    h = Dense(128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(inputs)
    h = Dense(7 * 7 * 64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Reshape((7, 7, 64))(h)
    h = Conv2DTranspose(32, 3, strides=2, activation='relu', padding='same', kernel_regularizer=REGULARIZER,
                        kernel_initializer=INITIALIZER_RELU)(h)  # 7 → 14
    outputs = Conv2DTranspose(1, 3, strides=2, activation='sigmoid', padding='same', kernel_regularizer=REGULARIZER,
                              kernel_initializer=INITIALIZER_SIGMOID)(h)
    return Model(inputs=inputs, outputs=outputs, name='decoder')


def get_encoder_v4(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the encoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''

    inputs = Input(shape=image_shape)
    h = Conv2D(32, 3, strides=2, activation='relu', padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(inputs)  # 28 → 14
    h = Conv2D(64, 3, strides=2, activation='relu', padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(h)  # 14 → 7
    h = Flatten()(h)
    h = Dense(128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(2 * latent_dim, kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    z_mean, z_log_var = ops.split(h, indices_or_sections=2, axis=-1)
    return Model(inputs=inputs, outputs=[z_mean, z_log_var], name='encoder')


def get_decoder_v5(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the decoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    # output size for Conv2D transpose:
    # out = (in − 1) x stride − 2 x padding + kernel_size + output_padding
    (img_h, img_w, img_c) = image_shape
    inputs = Input(shape=(latent_dim,))
    h = Dense(64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(inputs)
    h = Dense(128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(256, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(7 * 7 * 64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Reshape((7, 7, 64))(h)
    h = Conv2DTranspose(32, 3, strides=2, activation='relu', padding='same', kernel_regularizer=REGULARIZER,
                        kernel_initializer=INITIALIZER_RELU)(h)  # 7 → 14
    outputs = Conv2DTranspose(1, 3, strides=2, activation='sigmoid', padding='same', kernel_regularizer=REGULARIZER,
                              kernel_initializer=INITIALIZER_SIGMOID)(h)
    return Model(inputs=inputs, outputs=outputs, name='decoder')


def get_encoder_v5(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the encoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    inputs = Input(shape=image_shape)
    h = Conv2D(32, 3, strides=2, activation='relu', padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(inputs)  # 28 → 14
    h = Conv2D(64, 3, strides=2, activation='relu', padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(h)  # 14 → 7
    h = Flatten()(h)
    h = Dense(256, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(2 * latent_dim, kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    z_mean, z_log_var = ops.split(h, indices_or_sections=2, axis=-1)
    return Model(inputs=inputs, outputs=[z_mean, z_log_var], name='encoder')


def get_decoder_v6(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the decoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    # output size for Conv2D transpose:
    # out = (in − 1) x stride − 2 x padding + kernel_size + output_padding
    inputs = Input(shape=(latent_dim,))
    h = Dense(64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(inputs)
    h = Dense(128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(256, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(7 * 7 * 64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Reshape((7, 7, 64))(h)
    h = Conv2DTranspose(32, 3, strides=2, activation=None, use_bias=False, padding='same',
                        kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)  # 7 → 14
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Conv2DTranspose(1, 3, strides=2, activation=None, use_bias=False, padding='same',
                        kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_SIGMOID)(h)
    h = BatchNormalization()(h)
    outputs = keras.layers.Activation('sigmoid')(h)
    return Model(inputs=inputs, outputs=outputs, name='decoder')


def get_encoder_v6(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the encoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    inputs = Input(shape=image_shape)
    h = Conv2D(32, 3, strides=2, activation=None, use_bias=False, padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(inputs)  # 28 -> 14
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Conv2D(64, 3, strides=2, activation=None, use_bias=False, padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(h)  # 14 -> 7
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Flatten()(h)
    h = Dense(256, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(2 * latent_dim, kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    z_mean, z_log_var = ops.split(h, indices_or_sections=2, axis=-1)
    return Model(inputs=inputs, outputs=[z_mean, z_log_var], name='encoder')


def get_decoder_v7(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the decoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    # output size for Conv2D transpose:
    # out = (in − 1) x stride − 2 x padding + kernel_size + output_padding
    inputs = Input(shape=(latent_dim,))
    h = Dense(64, activation='relu', kernel_initializer=INITIALIZER_RELU, kernel_regularizer=REGULARIZER)(inputs)
    h = Dense(128, activation='relu', kernel_initializer=INITIALIZER_RELU, kernel_regularizer=REGULARIZER)(h)
    h = Dense(256, activation='relu', kernel_initializer=INITIALIZER_RELU, kernel_regularizer=REGULARIZER)(h)
    h = Dense(7 * 7 * 64, activation='relu', kernel_initializer=INITIALIZER_RELU, kernel_regularizer=REGULARIZER)(h)
    h = Reshape((7, 7, 64))(h)
    h = Conv2DTranspose(32, 3, strides=1, activation=None, use_bias=False, padding='same',
                        kernel_initializer=INITIALIZER_RELU, kernel_regularizer=REGULARIZER)(h)  # 7 → 14
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = UpSampling2D(2)(h)
    h = Conv2DTranspose(1, 3, strides=1, activation=None, use_bias=False, padding='same',
                        kernel_initializer=INITIALIZER_SIGMOID, kernel_regularizer=REGULARIZER)(h)
    h = BatchNormalization()(h)
    h = UpSampling2D(2)(h)
    outputs = keras.layers.Activation('sigmoid')(h)
    return Model(inputs=inputs, outputs=outputs, name='decoder')


def get_encoder_v7(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the encoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''

    inputs = Input(shape=image_shape)
    h = Conv2D(32, 3, strides=1, activation=None, use_bias=False, padding='same')(inputs)  # 28 → 14
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = MaxPool2D(2)(h)
    h = Conv2D(64, 3, strides=1, activation=None, use_bias=False, padding='same')(h)  # 14 → 7
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = MaxPool2D(2)(h)
    h = Flatten()(h)
    h = Dense(256, activation='relu')(h)
    h = Dense(128, activation='relu')(h)
    h = Dense(64, activation='relu')(h)
    h = BatchNormalization()(h)
    h = Dense(2 * latent_dim)(h)
    z_mean, z_log_var = ops.split(h, indices_or_sections=2, axis=-1)
    return Model(inputs=inputs, outputs=[z_mean, z_log_var], name='encoder')


def get_decoder_v8(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the decoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    # output size for Conv2D transpose:
    # out = (in − 1) x stride − 2 x padding + kernel_size + output_padding
    inputs = Input(shape=(latent_dim,))
    h = Dense(64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(inputs)
    h = Dense(128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(256, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(4 * 4 * 128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Reshape((4, 4, 128))(h)
    h = Conv2DTranspose(64, 2, strides=2, activation=None, use_bias=False, padding='same', output_padding=1,
                        kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)  # 7 → 14
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Conv2DTranspose(32, 3, strides=2, activation=None, use_bias=False, padding='same',
                        kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Conv2DTranspose(1, 3, strides=2, activation=None, use_bias=False, padding='same',
                        kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_SIGMOID)(h)
    h = BatchNormalization()(h)
    outputs = keras.layers.Activation('sigmoid')(h)
    return Model(inputs=inputs, outputs=outputs, name='decoder')


def get_encoder_v8(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the encoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    inputs = Input(shape=image_shape)
    h = Conv2D(32, 3, strides=2, activation=None, use_bias=False, padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(inputs)  # 28 -> 14
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Conv2D(64, 3, strides=2, activation=None, use_bias=False, padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(h)  # 14 -> 7
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Conv2D(128, 3, strides=2, activation=None, use_bias=False, padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(h)  # 14 -> 7
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Flatten()(h)
    h = Dense(256, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(2 * latent_dim, kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    z_mean, z_log_var = ops.split(h, indices_or_sections=2, axis=-1)
    return Model(inputs=inputs, outputs=[z_mean, z_log_var], name='encoder')


def get_decoder_v9(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the decoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    # output size for Conv2D transpose:
    # out = (in − 1) x stride − 2 x padding + kernel_size + output_padding
    inputs = Input(shape=(latent_dim,))
    h = Dense(128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(inputs)
    h = Dense(256, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(512, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(7 * 7 * 64, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Reshape((7, 7, 64))(h)
    h = Conv2DTranspose(32, 3, strides=2, activation=None, use_bias=False, padding='same',
                        kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)  # 7 → 14
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Conv2DTranspose(1, 3, strides=2, activation=None, use_bias=False, padding='same',
                        kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_SIGMOID)(h)
    h = BatchNormalization()(h)
    outputs = keras.layers.Activation('sigmoid')(h)
    return Model(inputs=inputs, outputs=outputs, name='decoder')


def get_encoder_v9(image_shape: Tuple[int, int, int], latent_dim: int):
    '''
    Get the encoder model

    Args:
        image_shape: The shape of the images - output tensors
        latent_dim: The latent dimension - dim of the inputs

    Returns: A keras model
    '''
    inputs = Input(shape=image_shape)
    h = Conv2D(32, 3, strides=2, activation=None, use_bias=False, padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(inputs)  # 28 -> 14
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Conv2D(64, 3, strides=2, activation=None, use_bias=False, padding='same', kernel_regularizer=REGULARIZER,
               kernel_initializer=INITIALIZER_RELU)(h)  # 14 -> 7
    h = BatchNormalization()(h)
    h = ReLU()(h)
    h = Flatten()(h)
    h = Dense(512, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(256, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(128, activation='relu', kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = Dense(2 * latent_dim, kernel_regularizer=REGULARIZER, kernel_initializer=INITIALIZER_RELU)(h)
    h = BatchNormalization()(h)
    z_mean, z_log_var = ops.split(h, indices_or_sections=2, axis=-1)
    return Model(inputs=inputs, outputs=[z_mean, z_log_var], name='encoder')


In [ ]:
from keras.api.callbacks import EarlyStopping


def train(encoder_getter, decoder_getter, model_class, batch_size: int = 1024, latent_dim: int = 2,
          num_mc_samples: int = 1):
    '''
    Helper function to facilitate train

    Args:
        encoder: A keras model that will serve as the encoder
        decoder: A keras model that will serve as the decoder
        batch_size: The batch size
        embedding_dim: The dimension of the latent space
        num_mc_samples: The number of MC samples to generate when computing the
        negative log likelihood.

    Returns: a trained model and the history.
    '''

    train, val, test = get_datasets(batch_size=batch_size)
    for data in train.take(1):
        image_shape = ops.shape(data)[1:]

    encoder = encoder_getter(image_shape=image_shape, latent_dim=latent_dim)
    decoder = decoder_getter(image_shape=image_shape, latent_dim=latent_dim)
    model = model_class(encoder=encoder, decoder=decoder, num_mc_samples=num_mc_samples)
    optimizer = keras.optimizers.Adam(learning_rate=1e-3, beta_1=0.9, beta_2=0.999, epsilon=1e-7)
    # optimizer = keras.optimizers.RMSprop(learning_rate=1e-3, momentum=0.001, epsilon=1e-6)
    model.compile(optimizer=optimizer, run_eagerly=keras.backend.backend() == 'tensorflow')
    _ = model(data)  # force build the model
    early_stopping = EarlyStopping(patience=10)
    history = model.fit(train, validation_data=val, epochs=1000, callbacks=[early_stopping])
    return history, model


def train_all_models(batch_size: int = 500):
    import pathlib
    import json
    backend = keras.backend.backend()
    _, _, test_data = get_datasets(batch_size)
    img_shape = (28, 28, 1)
    latent_dim = 2
    plots = pathlib.Path(f'./{backend}/validation_{batch_size}.png')
    fig, ax = plt.subplots()

    for i in range(10):
        root = pathlib.Path(f'./{backend}/version_{str(i)}_{str(batch_size)}')
        root.mkdir(parents=True, exist_ok=True)
        encoder = getattr(globals(), f'get_encoder_v{i}')
        decoder = getattr(globals(), f'get_decoder_v{i}')

        summary_path = root.joinpath('summary.txt')
        history_path = root.joinpath('history.json')
        model_weights_path = root.joinpath('model.weights.h5')
        with open(summary_path, 'w') as f:
            encoder(img_shape, latent_dim).summary(print_fn=lambda x: f.write(x + '\n'))
            decoder(img_shape, latent_dim).summary(print_fn=lambda x: f.write(x + '\n'))
        history, model = train(encoder_getter=encoder, decoder_getter=decoder, model_class=VAEDiagonal,
                               batch_size=batch_size)
        test_loss = model.evaluate(test_data)[0]
        ax.plot(history.history['val_loss'], label=f'Model {i}: {test_loss:.1f}')
        ax.legend()
        ax.set_title('Validation Loss')
        fig.savefig(plots)
        model.save_weights(model_weights_path, overwrite=True)
        with open(history_path, 'w') as f:
            json.dump(history.history, f)

#
# def do_test():
#     batch_size = 500
#     import pathlib
#     import matplotlib.pyplot as plt
#     backend = keras.backend.backend()
#
#     _, _, test_batch = get_datasets(batch_size)
#     for i in range(9):
#         import CW2.encoder_decoder as ed
#         history_path = pathlib.Path(f'./{backend}/version_{str(i)}_{str(batch_size)}/history.json')
#         path = pathlib.Path(f'./{backend}/version_{str(i)}_{str(batch_size)}/model.weights.h5')
#         encoder = getattr(ed, f'get_encoder_v{i}')((28, 28, 1), 2)
#         decoder = getattr(ed, f'get_decoder_v{i}')((28, 28, 1), 2)
#         model = VAEDiagonal(encoder=encoder, decoder=decoder, num_mc_samples=1)
#         model.build((None, 28, 28, 1))
#         model.load_weights(path)
#         model.compile()
#         res = model.evaluate(test_batch)[0]
#         print(f'version_{str(i)}_{str(batch_size)}: {res:.2f}')
#         with open(history_path, 'r') as f:
#             history = json.load(f)
#         plt.plot(history['val_loss'], linewidth=1, label=f'Model {i}: {res:.2f}')
#     plt.legend()
#     plt.show()

c) You should implement a `VAEFullCovariance` class to train a second model using the VAE algorithm, where the encoder network now parameterises a general (full covariance) multivariate Gaussian distribution over the 2-dimensional latent space. You should use the [Cholesky decomposition](https://en.wikipedia.org/wiki/Cholesky_decomposition) to parameterise the covariance matrix. The architectural design of the encoder and decoder network should otherwise be identical to the model you trained in part b). 

Your implementation should again follow the specifications:

* The `VAEFullCovariance` class should subclass from the `VAE` class above
* The latent space should have dimension 2
* Your implementation should be entirely using Keras
* The ELBO objective should be approximated using the second form of the SGVB estimator:
$$
\hat{\mathcal{L}}^B(\theta,\phi;x) := \frac{1}{L} \sum_{j=1}^L \log p_\theta(x\mid z^{(j)}) − D_{KL}(q_\phi(z\mid x) || p_\theta(z)),
$$
  * You should set $L=1$ in the model that you train (use a single Monte Carlo sample)
* The likelihood distribution $p_\theta(x\mid z^{(j)})$ should be chosen appropriately for the binary data, as in part b)
* Train your model using the high-level Keras API, and evaluate your trained model on the test set, clearly displaying the loss obtained on the test set.

**(6 marks)**

### Question 2 (Total 14 marks)

The importance weighted autoencoder (IWAE) objective is defined as

$$
\mathcal{L}_{IWAE}^k(\theta, \phi; x) := \mathbb{E}_{z_1,\ldots,z_k\sim q_\phi({z}\mid {x})} \left[ \log \frac{1}{k} \sum_{i=1}^k \frac{p_\theta({x}, {z_i})}{q_\phi({z_i}\mid {x})} \right],
$$

where $k\ge 1$ is a positive integer. The IWAE objective can be used as an optimisation objective instead of the VAE ELBO $\mathcal{L}_{VAE}$. The IWAE objective was introduced in the following paper, where it is shown that, similar to the VAE ELBO, $\mathcal{L}_{IWAE}^k$ is a lower bound on the marginal likelihood, and that $\log p_\theta(x) \ge \mathcal{L}_{IWAE}^{k+1} \ge \mathcal{L}_{IWAE}^k$. It is also shown (under certain conditions on $q_\phi(z\mid x)$) that $\mathcal{L}_{IWAE}^k \to \log p_\theta(x)$ as $k\to\infty$.

* Burda, Y., Grosse, R. and Salakhutdinov, R. (2015), "Importance Weighted Autoencoders", arXiv preprint, abs/1509.00519.

a) In Burda et al, it is proposed to use $\mathcal{L}_{IWAE}^k$ with $k=5000$ as a approximation of the true log-likelihood $\log p_\theta(x)$. 

Use this approximation of the log-likelihood to compute and display the negative log-likelihood (NLL) for the models trained in parts 1b) and 1c). Use a single Monte Carlo sample for each $z_1,\ldots,z_k$ (in other words, take a single sample of the vector $(z_1,\ldots,z_k)$) in the approximation of the expectation above.

_If you could not successfully implement the `VAEDiagonal` or `VAEFullCovariance` classes in questions 1b) and 1c), you can complete this question using the base `VAE` class provided for you, and a trained encoder/decoder as in the week 9 lecture notes._

**(4 marks)**

b) You should now train new encoder and decoder networks using the IWAE objective. The architectural design of the encoder and decoder networks should be identical to any previously trained models, with the encoder parameterising a fully factorised (diagonal) Gaussian for the approximate posterior $q_\phi(z \mid x)$ (as in the `VAE` and `VAEDiagonal` classes). You should use a `IWAEDiagonal` class to implement this (NB you may possibly have implemented this class in part a)).

Your implementation should follow the specifications:

* The `IWAEDiagonal` class should subclass from the `VAE` class or the `VAEDiagonal` class
* The latent space should have dimension 2
* Your implementation should be entirely using Keras
* The maximisation objective should be the IWAE objective $\mathcal{L}_{IWAE}^k$ with $k=50$. The expectation in the IWAE objective should be approximated using a single Monte Carlo sample for each $z_1,\ldots,z_k$
  * You should set $k=50$ in the model that you train, but your model class implementation should work for general $k$
* The likelihood distribution $p_\theta(x\mid z^{(j)})$ should be chosen appropriately for the binary data, as in question 1
* Train your model using the high-level Keras API, and evaluate your trained model on the test set, clearly displaying the loss obtained on the test set.

_Make sure to save your model as it will be used again in later questions._

**(3 marks)**

c) We define the distribution $q^k_{IWAE}(z\mid x)$ by

$$
q^k_{IWAE}(z\mid x) = \mathbb{E}_{z_2,\ldots,z_k\sim q_\phi(z\mid x)}\left[ \frac{p_\theta(x, z)}{\frac{1}{k} \left( \frac{p_\theta(x, z)}{q_\phi(z \mid x)} + \sum_{j=2}^k \frac{p_\theta(x, z_j)}{q_\phi(z_j \mid x)} \right)} \right].
$$

Prove that $q^k_{IWAE}(z\mid x)$ is a valid normalised distribution; that is, $\int_z q^k_{IWAE}(z\mid x) dz = 1$.

**(3 marks)**

d) It can be shown that (under certain conditions on $q_\phi(z\mid x)$) that the distribution $q^k_{IWAE}(z\mid x)$ converges to the true posterior distribution $p_\theta(z \mid x)$ as $k\to\infty$, in the sense that $D_{KL}(q^k_{IWAE}(z\mid x) \mid\mid p_\theta(z\mid x))$ converges to zero.

It can also be shown that as $k\to\infty$, we have that $\left|\mathcal{L}^k_{IWAE}(\theta,\phi; x) - \mathcal{L}_{VAE}(\theta,\phi; x)[q^k_{IWAE}]\right| \to 0$, where

$$
\mathcal{L}_{VAE}(\theta,\phi; x)[q^k_{IWAE}] := \mathbb{E}_{z\sim q^k_{IWAE}({z}\mid {x})} \left[ \log p_\theta({x}, {z}) - \log q^k_{IWAE}({z}\mid {x}) \right]
$$

That is, the IWAE objective converges to the VAE ELBO objective where the variational posterior $q_\phi(z \mid x)$ is replaced with $q^k_{IWAE}(z\mid x)$. 

Using your trained encoder and decoder networks from part b), display some test images and their reconstructions using i) $q_\phi(z \mid x)$ and ii) $q^k_{IWAE}(z\mid x)$. The reconstructions displayed should be the mean of the likelihood distribution $p_\theta(x \mid z)$, where $z$ is a sample from the specified posterior distribution ($q_\phi(z \mid x)$ or $q^k_{IWAE}(z\mid x)$). Set $k=50$ for the distribution $q^k_{IWAE}(z\mid x)$, as in part b).

The following algorithm can be used to sample from $q^k_{IWAE}(z\mid x)$:

<center><img src="figures/sampling_algorithm.png" alt="Sampling algorithm" style="width: 500px;"/></center>

**(4 marks)**

#### Question 3 (Total 9 marks)

In this question you will investigate the strength of the learning signal for the models trained with either $\mathcal{L}_{VAE}$ or $\mathcal{L}^k_{IWAE}$. The quantity that you will use to measure this is the signal-to-noise ratio of the gradients used in training. First we define (for the decoder parameters $\theta$):

$$
\begin{align}
\Delta_{VAE}(\theta) &:= \frac{\partial \mathcal{L}_{VAE}(\theta, \phi; x)}{\partial \theta},\\
\Delta^k_{IWAE}(\theta) &:= \frac{\partial \mathcal{L}^k_{IWAE}(\theta, \phi; x)}{\partial \theta},
\end{align}
$$
and similarly for the encoder parameters $\phi$. Then the signal-to-noise ratio is given by

$$
\begin{align}
SNR_{VAE}(\theta) &:= \left| \frac{\mathbb{E}[\Delta_{VAE}(\theta)]}{\sigma[\Delta_{VAE}(\theta)]} \right|,\\[1.5ex]
SNR_{IWAE}(\theta) &:= \left| \frac{\mathbb{E}[\Delta_{IWAE}(\theta)]}{\sigma[\Delta_{IWAE}(\theta)]} \right|,
\end{align}
$$
where $\sigma[\cdot]$ denotes the standard deviation of a random variable. The signal-to-noise ratio is defined similarly for the encoder parameters $\phi$. The expectation and standard deviation in the above definition is with respect to the Monte Carlo samples drawn from the posterior $q_\phi(z\mid x)$ in the definitions of $\mathcal{L}_{VAE}$ and $\mathcal{L}^k_{IWAE}$. For this question you should assume that the encoder parameterises a diagonal Gaussian for the approximate posterior (as in the `VAEDiagonal` and `IWAEDiagonal` classes). 

_If you have not been able to successfully implement the `VAEDiagonal` or `IWAEDiagonal` classes in questions 1b) and 2b), you can complete all of question 3 question using the base `VAE` class provided for you, and a trained encoder/decoder as in the week 9 lecture notes._

a) Write a function `compute_snr` that computes estimates for the signal-to-noise ratio for a given list of trainable Variables, and given a single data example. Your function should compute estimates for either $SNR_{VAE}$ or $SNR_{IWAE}$ by estimating the expectation and standard deviation with Monte Carlo samples. Your function should take the following arguments:

* `encoder` and `decoder` network objects. You can assume that the encoder parameterises a diagonal Gaussian for the approximate posterior
* A list of Keras Variable objects (that belong to the encoder or decoder networks)
* An option to compute $SNR_{VAE}$ or $SNR_{IWAE}$
  * If computing $SNR_{IWAE}$, the function will also require $k$ as an argument
* Number of Monte Carlo samples to estimate the expectation and standard deviation in the definition of the $SNR$
* A `data` example that the loss is computed on

The SNR is defined for each parameter. Your function should return a list of SNR values. This list will contain Tensors with the same shape as in the list of trainable Variables input to the function.

_Hint: you may need to add a small value to the computed standard deviation for numerical stability._

**(6 marks)**

b) Using your `compute_snr` function from part a), compute $SNR_{VAE}$ and $SNR_{IWAE}$ (with $k=50$) for the encoder trainable variables $\phi$, and for the decoder trainable variables $\theta$. The SNR values should be averaged over a batch of data drawn from the training Dataset.

Display histogram plots (one for $\phi$ and one for $\theta$) showing the distribution of SNR values under the $\mathcal{L}_{VAE}$ loss and $\mathcal{L}_{IWAE}^{50}$ loss.

**(3 marks)**

#### Question 4 (Total 10 marks)

Provide a separate PDF report with an account of the experiments you have run in this assessment. Your report should include details of the results from all experiments, your choice of architecture for the encoder and decoder networks, and any other choices you have made throughout the development of the models. If you encountered difficulties to successfully train the model(s) you should report these, and if possible suggest what you think might be potential reasons for these difficulties. 

Your report should also include your interpretation and analysis of the results, and what you think they might suggest about training inference and generative networks on the binarised MNIST dataset using the VAE or IWAE training objective.

Marks will be awarded for presentation and clarity. Your report should be no more than 2 pages, excluding any references and figures (which can be included in an appendix).

**(10 marks)**